#### 0. Import module

In [1]:
import pandas as pd
import numpy as np
import re

#### 1. Data Overview

In [2]:
df = pd.read_csv("vietnam_housing_dataset.csv")
print(df.shape)
df.head(10)

(30229, 12)


,Address,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,Furniture state,Price
0,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",84.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,Have certificate,NaN,8.60
1,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",60.0,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,7.50
2,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",90.0,6.0,13.0,Đông - Bắc,Đông - Bắc,5.0,NaN,NaN,Sale contract,NaN,8.90
3,"Đường Nguyễn Văn Khối, Phường 11, Gò Vấp, Hồ C...",54.0,NaN,3.5,Tây - Nam,Tây - Nam,2.0,2.0,3.0,Have certificate,Full,5.35
4,"Đường Quang Trung, Phường 8, Gò Vấp, Hồ Chí Minh",92.0,NaN,NaN,Đông - Nam,Đông - Nam,2.0,4.0,4.0,Have certificate,Full,6.90
5,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",91.0,7.0,13.0,Tây - Bắc,NaN,NaN,NaN,NaN,Have certificate,NaN,9.80
6,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",64.0,4.0,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,7.20
7,"Dự án Him Lam Thường Tín, Huyện Thường Tín, Hà...",74.0,5.0,18.0,Nam,Nam,5.0,4.0,5.0,Have certificate,NaN,9.90
8,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",48.0,NaN,NaN,NaN,NaN,5.0,6.0,NaN,NaN,Basic,5.70
9,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",91.0,7.0,NaN,NaN,NaN,5.0,NaN,NaN,Have certificate,NaN,9.50


In [3]:
print("=== THÔNG TIN CỘT ===")
print(df.info())

print("\n=== MISSING VALUES (%) ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(
    pd.DataFrame({"Missing": missing, "Missing %": missing_pct}).sort_values(
        "Missing %", ascending=False
    )
)

print("\n=== THỐNG KÊ MÔ TẢ ===")
df.describe()

=== THÔNG TIN CỘT ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30229 entries, 0 to 30228
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Address            30229 non-null  object 
 1   Area               30229 non-null  float64
 2   Frontage           18665 non-null  float64
 3   Access Road        16932 non-null  float64
 4   House direction    8990 non-null   object 
 5   Balcony direction  5246 non-null   object 
 6   Floors             26626 non-null  float64
 7   Bedrooms           25067 non-null  float64
 8   Bathrooms          23155 non-null  float64
 9   Legal status       25723 non-null  object 
 10  Furniture state    16110 non-null  object 
 11  Price              30229 non-null  float64
dtypes: float64(7), object(5)
memory usage: 2.8+ MB
None

=== MISSING VALUES (%) ===
                   Missing  Missing %
Balcony direction    24983      82.65
House direction      21239      70.2

,Area,Frontage,Access Road,Floors,Bedrooms,Bathrooms,Price
count,30229.000000,18665.000000,16932.000000,26626.000000,25067.000000,23155.000000,30229.000000
mean,68.498741,5.361692,7.853800,3.410426,3.511030,3.346837,5.872078
std,48.069835,4.346174,7.451313,1.328897,1.309116,1.400181,2.211877
min,3.100000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,40.000000,4.000000,4.000000,2.000000,3.000000,2.000000,4.200000
50%,56.000000,4.500000,6.000000,3.000000,3.000000,3.000000,5.900000
75%,80.000000,5.000000,10.000000,4.000000,4.000000,4.000000,7.500000
max,595.000000,77.000000,85.000000,10.000000,9.000000,9.000000,11.500000


#### 1. 2 Data Assessment & Preprocessing Approach (Đánh giá dữ liệu)

**2.1 Nhận xét**
Sau khi quan sát cấu trúc của bộ dữ liệu bất động sản Việt Nam, đây là các vấn đề chính:

**Giá trị thiếu (Missing Values**:
* Cột `Area` và `Price` có 30,229 bản ghi (đầy đủ nhất).
* Trong khi đó, `Access Road` chỉ có 16,932 và `Frontage` chỉ có 18,665 bản ghi. Tức là có tới gần **45%** dữ liệu trống ở hai cột này. Điền giá trị 0 vì không có thông tin hoặc giá trị bằng 0 thường cũng không được quan tâm.
* Các biến về *số phòng tắm, phòng ngủ, số tầng*: thiếu chiếm hơn 10% số lượng dữ liệu nên không loại bỏ, xử lý bằng impute (trung vị theo tỉnh hoặc trung bình cả nước nếu trong tỉnh không có)
* Các thông tin như *hướng nhà, hướng ban công* thường mang tính chất là thông tin bổ sung thêm hoặc sẽ không đóng góp nhiều vào vai trò phân tích chính trong project này. Hơn thế nữa, phần trăm giá trị thiếu của 2 biến này cực lớn, chiếm tới trên 70%, nếu xóa sẽ làm mất đi gần như toàn bộ thông tin của bộ dữ liệu.
* Các biến *hướng nhà, hướng ban công* hay *tình trạng pháp lý*: Đều không được phép đoán. Nên sẽ thay thế các giá trị trống này bằng *"Chưa có thông tin"* / *Unknown*
* Thông tin *tình trạng nội thất*: chưa có thông tin nội thất hoặc chưa có nội thất cũng không cung cấp được thông tin quan trọng, nên người mua sẽ không dựa vào thông tin này để định giá. Hơn nữa giá trị này chiếm hơn 10% dataset nên sẽ thay thế các giá trị thống này bằng *"Chưa có thông tin"* / *Unknown*.
  
**Địa chỉ không đồng nhất:** Cột địa chỉ thường là một chuỗi dài (Xã, Huyện, Tỉnh).Cần tách ra thành các cấp hành chính riêng biệt trước khi phân tích.

<!-- **Dữ liệu nhiễu (Outliers):** Các dữ liệu ngoại lệ, quá lớn hoặc quả nhỏ vẫn phù hợp trong ngữ cảnh bất động sản này. -->
  
**2.2 Đề xuất tiền xử lý & Tạo đặc trưng (Feature Engineering)**

**Làm sạch (Cleaning)**
* **Chuẩn hóa**
    * Phân rã cột địa chỉ ra thành các cấp hành chính.
* **Xử lý Missing Values:** 
    * *Với thuộc tính Numerical*: điền giá trị 0
    * *Với thuộc tính Categorical*:Thay các giá trị còn thiếu bằng giá trị `"Chưa có thông tin"`/ *Unknown*

**Tạo đặc trưng mới (Feature Engineering):**
* `Giá trên m2 (Price per m2)`: `price / area. Là thông tin quan trọng mà người mua thường chú ý
* **Trích xuất Địa chỉ:** Tách cột địa chỉ để lấy Quận/Huyện và Tỉnh/Thành phố.
* **Phân loại loại hình:** Dựa vào mô tả để phân loại là "Nhà riêng", "Chung cư", hay "Đất nền".
* `Is_project`:Xem đó là *dự án* hay *nhà phố* (nếu có xuất hiện tên dự án thì đó là dự án.)
* `Has_certificate`: Xem có đầy đủ pháp lý không
* `Area_Group`: Phân ra thành 5 nhóm diện tích

#### 2. Data Prepocessing

#### 2.1 Data Cleaning

##### 1.Phân rã địa chỉ

In [4]:
##  ----------------------------------- CODE CŨ ------------------------
# def parse_address(addr):
#     if pd.isna(addr):
#         return pd.Series(
#             {"Project": None, "Ward_Street": None, "District": None, "Province": None}
#         )

#     parts = [p.strip() for p in addr.split(",")]

#     result = {"Project": None, "Ward_Street": None, "District": None, "Province": None}
    
#     # 2 vị trí cuối: tỉnh, huyện
#     if len(parts) >= 1:
#         result["Province"] = parts[-1]
#     if len(parts) >= 2:
#         result["District"] = parts[-2]

#     # Vị trí đầu: dự án hoặc không
#     if len(parts) >= 1:
#         first = parts[0]
#         if first.lower().startswith("dự án"):
#             result["Project"] = first
#             middle = parts[1:-2]  # phần giữa sau dự án
#         else:
#             middle = parts[0:-2]  # phần giữa tính từ đầu

#         # Gom phần giữa vào Ward_Street
#         if middle:
#             result["Ward_Street"] = ", ".join(middle)

#     return pd.Series(result)


# addr_parsed = df["Address"].apply(parse_address)
# df = pd.concat([df, addr_parsed], axis=1)
# df.head(5) 

In [5]:
def parse_address(addr):
    result = {
        "Project": np.nan, 
        "Street": np.nan, 
        "Ward": np.nan, 
        "District": np.nan, 
        "Province": np.nan
    }
    
    if pd.isna(addr) or str(addr).strip() == "":
        return pd.Series(result)

    parts = [p.strip() for p in str(addr).split(",")]
    n_parts = len(parts)

    # 1. Định danh 2 vị trí cuối: Province và District
    if n_parts >= 1:
        result["Province"] = parts[-1]
    if n_parts >= 2:
        result["District"] = parts[-2]

    # 2. Xử lý phần tử đầu (Project) và xác định phần "Middle"
    if n_parts >= 1:
        first = parts[0]
        if first.lower().startswith("dự án"):
            result["Project"] = first
            middle_parts = parts[1:-2] if n_parts > 2 else []
        else:
            middle_parts = parts[0:-2] if n_parts > 1 else []

        # 3. Dùng Regex để tách Ward và Street từ phần ở giữa
        if middle_parts:
            middle_str = ", ".join(middle_parts)
            
            # Regex tìm Phường/Xã
            ward_match = re.search(r'(Phường|Xã)\s+([^,]+)', middle_str, re.IGNORECASE)
            
            if ward_match:
                result["Ward"] = ward_match.group(0).strip()
                # Street là phần còn dư lại sau khi bỏ ward
                street_clean = middle_str.replace(ward_match.group(0), "").strip().strip(",").strip()
                result["Street"] = street_clean if street_clean else None
            else:
                # Nếu không bắt được pattern Ward, coi toàn bộ là Street
                result["Street"] = middle_str

    return pd.Series(result)

# Áp dụng vào DataFrame
addr_parsed = df["Address"].apply(parse_address)
df = pd.concat([df, addr_parsed], axis=1)


# Hiển thị kiểm tra
print(df[["Address", "Project", "Street", "Ward", "District", "Province"]].head(5))

                                             Address  \
0  Dự án The Empire - Vinhomes Ocean Park 2, Xã L...   
1  Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...   
2  Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...   
3  Đường Nguyễn Văn Khối, Phường 11, Gò Vấp, Hồ C...   
4   Đường Quang Trung, Phường 8, Gò Vấp, Hồ Chí Minh   

                                    Project                 Street  \
0  Dự án The Empire - Vinhomes Ocean Park 2                   None   
1   Dự án The Crown - Vinhomes Ocean Park 3                   None   
2   Dự án The Crown - Vinhomes Ocean Park 3                   None   
3                                       NaN  Đường Nguyễn Văn Khối   
4                                       NaN      Đường Quang Trung   

           Ward   District     Province  
0  Xã Long Hưng  Văn Giang     Hưng Yên  
1  Xã Nghĩa Trụ  Văn Giang     Hưng Yên  
2  Xã Nghĩa Trụ  Văn Giang     Hưng Yên  
3     Phường 11     Gò Vấp  Hồ Chí Minh  
4      Phường 8     Gò Vấp  Hồ C

##### 2.Xử lý giá trị ngoại lai 
***Tuy nhiên trong ngữ cảnh bất động sản này thì không biết có nên bỏ hay không***
- 1. Ở chỗ diện tích- diện tích max lớn, nhưng vẫn hợp lý, xử dụng IQR thì rất nhiều giá trị hợp lệ bị mất
- 2. Ở price, giá rẻ hay mắc là tùy thị trường ở tùy tính, không nên loại bỏ nếu nó không null.
    -> Vì nếu loại bỏ theo IQR thì rất nhiều giá trị hợp lý bị loại bỏ.
- 3. Frontage thì nên loại bỏ giá trị > 40 hoặc ép về khoảng 0 - 40 bằng cách chia cho lũy thừa 10 (vì thường không có lối đi nào cao hơn đường cao tốc)

In [6]:
def flag_outliers_iqr(series, factor=3.0):
    """Dùng IQR với factor rộng hơn (3.0) để chỉ flag outlier cực đoan"""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return (series < lower) | (series > upper)


outlier_cols = ["Area", "Frontage", "Access Road", "Price"]

for col in outlier_cols:
    mask = flag_outliers_iqr(df[col].dropna())
    n_outliers = mask.sum()
    print(f"{col}: {n_outliers} outliers ({n_outliers/len(df)*100:.2f}%)")
    if n_outliers > 0:
        print(df.loc[df[col].notna()][mask][[col, "Address"]].head(5))
        print()

Area: 619 outliers (2.05%)
      Area                                            Address
16   300.0  Dự án Khu đô thị Phương Đông, Xã Đông Xá, Vân ...
24   369.0  Dự án Vườn Vua Resort & Villas, Đường 316, Xã ...
41   378.0                  Xã Lại Thượng, Thạch Thất, Hà Nội
78   368.0  Dự án Vườn Vua Resort & Villas, Đường Trung Th...
208  272.0                  Xã Bình Minh, Trảng Bom, Đồng Nai

Frontage: 1134 outliers (3.75%)
    Frontage                                            Address
16      15.0  Dự án Khu đô thị Phương Đông, Xã Đông Xá, Vân ...
24      15.0  Dự án Vườn Vua Resort & Villas, Đường 316, Xã ...
37      17.0  Dự án Hoàng Nam Uyên Hưng, Đường Tố Hữu, Phườn...
48      10.0  Đường Quốc Lộ 17B, Phường Minh Tân, Kinh Môn, ...
73      10.0  Dự án NovaWorld Phan Thiết, Đường Lạc Long Quâ...

Access Road: 468 outliers (1.55%)
    Access Road                                            Address
16         32.0  Dự án Khu đô thị Phương Đông, Xã Đông Xá, Vân ...
37         36.0

##### 3.Xử lý giá trị thiếu 

###### Categorical
-  Categorical: giữ NaN (không đoán mò hướng nhà / pháp lý)

In [7]:
# Chuẩn hóa text: strip whitespace, title case
cat_cols = ["House direction", "Balcony direction", "Legal status", "Furniture state"]

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()
    # df[col] = df[col].replace("nan", np.nan)
    df[col] = df[col].replace("nan", "Unknown")

# Kiểm tra các giá trị unique
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- House direction ---
House direction
Unknown       21239
Đông - Nam     1916
Đông - Bắc     1180
Tây - Nam      1152
Tây - Bắc      1125
Nam            1042
Đông            952
Bắc             832
Tây             791
Name: count, dtype: int64

--- Balcony direction ---
Balcony direction
Unknown       24983
Đông - Nam     1087
Nam             701
Đông - Bắc      670
Tây - Nam       646
Tây - Bắc       640
Đông            577
Bắc             474
Tây             451
Name: count, dtype: int64

--- Legal status ---
Legal status
Have certificate    24774
Unknown              4506
Sale contract         949
Name: count, dtype: int64

--- Furniture state ---
Furniture state
Unknown    14119
Full       10591
Basic       5519
Name: count, dtype: int64


###### Numerical
- Chiến lược impute

In [8]:
# Thống kê mô tả
numeric_cols = [
    "Area",
    "Frontage",
    "Access Road",
    "Floors",
    "Bedrooms",
    "Bathrooms",
    "Price",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_cols].dtypes)
print(df[numeric_cols].describe())

Area           float64
Frontage       float64
Access Road    float64
Floors         float64
Bedrooms       float64
Bathrooms      float64
Price          float64
dtype: object
               Area      Frontage   Access Road        Floors      Bedrooms  \
count  30229.000000  18665.000000  16932.000000  26626.000000  25067.000000   
mean      68.498741      5.361692      7.853800      3.410426      3.511030   
std       48.069835      4.346174      7.451313      1.328897      1.309116   
min        3.100000      1.000000      1.000000      1.000000      1.000000   
25%       40.000000      4.000000      4.000000      2.000000      3.000000   
50%       56.000000      4.500000      6.000000      3.000000      3.000000   
75%       80.000000      5.000000     10.000000      4.000000      4.000000   
max      595.000000     77.000000     85.000000     10.000000      9.000000   

          Bathrooms         Price  
count  23155.000000  30229.000000  
mean       3.346837      5.872078  
std  

In [9]:
# Xử lý giá trị Numerical thiếu bằng impute
# --- Chiến lược impute ---
# Số: dùng MEDIAN theo Province (tránh bị kéo bởi outlier)
for col in ["Area", "Frontage", "Access Road", "Floors", "Bedrooms", "Bathrooms"]:
    df[col] = df.groupby("Province")[col].transform(lambda x: x.fillna(x.median()))
    # Nếu vẫn còn NaN (tỉnh chỉ có 1 dòng), fallback về median toàn bộ
    df[col] = df[col].fillna(df[col].median())

print("Missing sau impute:")
print(df[numeric_cols].isnull().sum())

C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarnin

Missing sau impute:
Area           0
Frontage       0
Access Road    0
Floors         0
Bedrooms       0
Bathrooms      0
Price          0
dtype: int64


C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarnin

#### 2.2 Feature Engineering

In [10]:
# Giá trên m2
df["Price_per_m2"] = (df["Price"] / df["Area"]).round(4)

# Loại bất động sản: dự án hay nhà phố
df["Is_Project"] = df["Project"].notna().astype(int)

# Nhóm diện tích
df["Area_group"] = pd.cut(
    df["Area"],
    bins=[0, 50, 80, 120, 200, float("inf")],
    labels=["<50m²", "50-80m²", "80-120m²", "120-200m²", ">200m²"],
)

# Có đầy đủ pháp lý không
df["Has_certificate"] = (df["Legal status"] == "Have certificate").astype(int)

print(
    df[
        ["Price", "Area", "Price_per_m2", "Is_Project", "Area_group", "Has_certificate"]
    ].head(10)
)

   Price  Area  Price_per_m2  Is_Project Area_group  Has_certificate
0   8.60  84.0        0.1024           1   80-120m²                1
1   7.50  60.0        0.1250           1    50-80m²                0
2   8.90  90.0        0.0989           1   80-120m²                0
3   5.35  54.0        0.0991           0    50-80m²                1
4   6.90  92.0        0.0750           0   80-120m²                1
5   9.80  91.0        0.1077           1   80-120m²                1
6   7.20  64.0        0.1125           1    50-80m²                0
7   9.90  74.0        0.1338           1    50-80m²                1
8   5.70  48.0        0.1188           1      <50m²                0
9   9.50  91.0        0.1044           1   80-120m²                1


#### 3. Save to file & Checking

In [11]:
n_before = len(df)
df = df.drop_duplicates()
n_after = len(df)
print(f"Removed {n_before - n_after} duplicate rows. Remaining: {n_after}")

Removed 43 duplicate rows. Remaining: 30186


In [12]:
print("=== FINAL SHAPE ===")
print(df.shape)

print("\n=== MISSING VALUES SAU CLEANING ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n=== SAMPLE ===")
df.head(5)

=== FINAL SHAPE ===
(30186, 21)

=== MISSING VALUES SAU CLEANING ===
Project     26549
Street       3780
Ward          734
District        3
dtype: int64

=== SAMPLE ===


,Address,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,...,Price,Project,Street,Ward,District,Province,Price_per_m2,Is_Project,Area_group,Has_certificate
0,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",84.0,5.0,13.0,Unknown,Unknown,4.0,6.0,4.0,Have certificate,...,8.60,Dự án The Empire - Vinhomes Ocean Park 2,None,Xã Long Hưng,Văn Giang,Hưng Yên,0.1024,1,80-120m²,1
1,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",60.0,5.0,13.0,Unknown,Unknown,5.0,6.0,4.0,Unknown,...,7.50,Dự án The Crown - Vinhomes Ocean Park 3,None,Xã Nghĩa Trụ,Văn Giang,Hưng Yên,0.1250,1,50-80m²,0
2,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",90.0,6.0,13.0,Đông - Bắc,Đông - Bắc,5.0,6.0,4.0,Sale contract,...,8.90,Dự án The Crown - Vinhomes Ocean Park 3,None,Xã Nghĩa Trụ,Văn Giang,Hưng Yên,0.0989,1,80-120m²,0
3,"Đường Nguyễn Văn Khối, Phường 11, Gò Vấp, Hồ C...",54.0,4.1,3.5,Tây - Nam,Tây - Nam,2.0,2.0,3.0,Have certificate,...,5.35,NaN,Đường Nguyễn Văn Khối,Phường 11,Gò Vấp,Hồ Chí Minh,0.0991,0,50-80m²,1
4,"Đường Quang Trung, Phường 8, Gò Vấp, Hồ Chí Minh",92.0,4.1,6.0,Đông - Nam,Đông - Nam,2.0,4.0,4.0,Have certificate,...,6.90,NaN,Đường Quang Trung,Phường 8,Gò Vấp,Hồ Chí Minh,0.0750,0,80-120m²,1


##### Standarize Province

In [13]:
# 1. Danh sách các tỉnh/thành chuẩn (trích xuất từ danh sách của bạn)
valid_provinces = [
    'Hà Nội', 'Hồ Chí Minh', 'Hưng Yên', 'Quảng Ninh', 'Bình Dương', 'Phú Thọ',
    'Hải Dương', 'Long An', 'Kiên Giang', 'Bình Thuận', 'Hải Phòng', 'Hà Nam',
    'Bà Rịa Vũng Tàu', 'Đắk Lắk', 'Bắc Ninh', 'Thanh Hóa', 'Khánh Hòa', 'Đà Nẵng',
    'Thái Nguyên', 'Quảng Trị', 'Đồng Nai', 'Tuyên Quang', 'Ninh Thuận', 'Nghệ An',
    'Cần Thơ', 'Thái Bình', 'Bắc Giang', 'Lâm Đồng', 'Trà Vinh', 'Bình Phước',
    'Quảng Nam', 'Hòa Bình', 'Thừa Thiên Huế', 'Sơn La', 'Bình Định', 'Vĩnh Phúc',
    'Tây Ninh', 'Tiền Giang', 'Hà Tĩnh', 'Lạng Sơn', 'Lào Cai', 'Điện Biên',
    'Gia Lai', 'Phú Yên', 'Bến Tre', 'An Giang', 'Quảng Ngãi', 'Hà Giang',
    'Nam Định', 'Sóc Trăng', 'Ninh Bình', 'Yên Bái', 'Cà Mau', 'Bạc Liêu',
    'Vĩnh Long', 'Đồng Tháp', 'Hậu Giang', 'Quảng Bình', 'Kon Tum'
]

# 2. Làm sạch thô: Bỏ dấu chấm ở cuối và khoảng trắng
df['Province'] = df['Province'].str.strip().str.rstrip('.')

# 3. Mapping các biến thể về tên chuẩn
province_map = {
    'TP. HCM': 'Hồ Chí Minh', 
    'TPHCM': 'Hồ Chí Minh', 
    'TpHCM': 'Hồ Chí Minh', 
    'TP Hồ Chí Minh': 'Hồ Chí Minh',
    'Hồ Chí Mính': 'Hồ Chí Minh',
    'HN': 'Hà Nội',
    'Hồ Chí Minh giá 2tỷ380': 'Hồ Chí Minh'
}
df['Province'] = df['Province'].replace(province_map)

# 4. Xử lý trường hợp có tên tỉnh nằm trong chuỗi nhiễu 
# (Ví dụ: "Quảng Ninh (Ngã 3...)" -> "Quảng Ninh")
def extract_valid_province(p):
    if pd.isna(p): return None
    for valid in valid_provinces:
        if valid in p:
            return valid
    return None

In [14]:
#  Lưu lại số lượng dòng trước khi lọc
initial_count = len(df)

# (Thực hiện các bước làm sạch và hàm extract_valid_province như trên...)
df['Province'] = df['Province'].apply(extract_valid_province)

# 5. Thực hiện loại bỏ các dòng không có tỉnh hợp lệ
df_cleaned = df.dropna(subset=['Province'])

# 5.1. Tính toán số lượng bị loại bỏ
final_count = len(df_cleaned)
removed_count = initial_count - final_count

print(f"--- KẾT QUẢ LỌC TỈNH THÀNH ---")
print(f"Số dòng ban đầu: {initial_count}")
print(f"Số dòng bị loại bỏ (do dữ liệu rác/thiếu tỉnh): {removed_count}")
print(f"Số dòng còn lại: {final_count}")
print(f"Tỷ lệ dữ liệu rác: {removed_count / initial_count * 100:.2f}%")

# Cập nhật lại df chính
df = df_cleaned

# 6. Kiểm tra lại danh sách tỉnh đã sạch
print("Danh sách tỉnh sau khi làm sạch:")
print(df['Province'].unique())

--- KẾT QUẢ LỌC TỈNH THÀNH ---
Số dòng ban đầu: 30186
Số dòng bị loại bỏ (do dữ liệu rác/thiếu tỉnh): 13
Số dòng còn lại: 30173
Tỷ lệ dữ liệu rác: 0.04%
Danh sách tỉnh sau khi làm sạch:
['Hưng Yên' 'Hồ Chí Minh' 'Hà Nội' 'Quảng Ninh' 'Bình Dương' 'Phú Thọ'
 'Hải Dương' 'Long An' 'Kiên Giang' 'Bình Thuận' 'Hải Phòng' 'Hà Nam'
 'Bà Rịa Vũng Tàu' 'Đắk Lắk' 'Bắc Ninh' 'Thanh Hóa' 'Khánh Hòa' 'Đà Nẵng'
 'Thái Nguyên' 'Quảng Trị' 'Đồng Nai' 'Tuyên Quang' 'Ninh Thuận' 'Nghệ An'
 'Cần Thơ' 'Thái Bình' 'Bắc Giang' 'Lâm Đồng' 'Trà Vinh' 'Bình Phước'
 'Quảng Nam' 'Hòa Bình' 'Thừa Thiên Huế' 'Sơn La' 'Bình Định' 'Vĩnh Phúc'
 'Tây Ninh' 'Tiền Giang' 'Hà Tĩnh' 'Lạng Sơn' 'Lào Cai' 'Điện Biên'
 'Gia Lai' 'Phú Yên' 'Bến Tre' 'An Giang' 'Quảng Ngãi' 'Hà Giang'
 'Nam Định' 'Sóc Trăng' 'Ninh Bình' 'Yên Bái' 'Cà Mau' 'Bạc Liêu'
 'Vĩnh Long' 'Đồng Tháp' 'Hậu Giang' 'Quảng Bình' 'Kon Tum']


##### Save to file

In [15]:
# Lưu file đã clean
df.to_csv("vietnam_housing_dataset_cleaned.csv", index=False, encoding="utf-8-sig")

##### Checking

In [16]:
df = pd.read_csv("vietnam_housing_dataset_cleaned.csv")
print(df.shape)

df["Province"].dropna().unique().tolist()

(30173, 21)


['Hưng Yên',
 'Hồ Chí Minh',
 'Hà Nội',
 'Quảng Ninh',
 'Bình Dương',
 'Phú Thọ',
 'Hải Dương',
 'Long An',
 'Kiên Giang',
 'Bình Thuận',
 'Hải Phòng',
 'Hà Nam',
 'Bà Rịa Vũng Tàu',
 'Đắk Lắk',
 'Bắc Ninh',
 'Thanh Hóa',
 'Khánh Hòa',
 'Đà Nẵng',
 'Thái Nguyên',
 'Quảng Trị',
 'Đồng Nai',
 'Tuyên Quang',
 'Ninh Thuận',
 'Nghệ An',
 'Cần Thơ',
 'Thái Bình',
 'Bắc Giang',
 'Lâm Đồng',
 'Trà Vinh',
 'Bình Phước',
 'Quảng Nam',
 'Hòa Bình',
 'Thừa Thiên Huế',
 'Sơn La',
 'Bình Định',
 'Vĩnh Phúc',
 'Tây Ninh',
 'Tiền Giang',
 'Hà Tĩnh',
 'Lạng Sơn',
 'Lào Cai',
 'Điện Biên',
 'Gia Lai',
 'Phú Yên',
 'Bến Tre',
 'An Giang',
 'Quảng Ngãi',
 'Hà Giang',
 'Nam Định',
 'Sóc Trăng',
 'Ninh Bình',
 'Yên Bái',
 'Cà Mau',
 'Bạc Liêu',
 'Vĩnh Long',
 'Đồng Tháp',
 'Hậu Giang',
 'Quảng Bình',
 'Kon Tum']